#### Preamble for `CoLab`

To use this notebook (if you haven't already) you can first save a copy to your local drive by clicking `File > Save a Copy in Drive` and run on that copy.

_Note_: `Colab` is a really handy way to test and try `strauss`, though it will generally run and display audio more slowly than running on your local machine. For a more responsive experience, why not install `strauss` locally, following the instructions [on the Github](https://github.com/james-trayford/strauss)

Run these cells, so that the notebook functions on the _Google_ `Colab` platform:

### <u> Some Set-*up* </u>

In [ ]:
%pip --quiet install git+https://github.com/james-trayford/strauss.git@v2

Grab a non-pitched sample kit

In [ ]:
!sudo apt-get install unar > /dev/null
!wget https://rhythm-lab.com/sstorage/44/2015/02/rhythm-lab.com_scratch-oneshot-fx-pack.rar > /dev/null
!unar rhythm-lab.com_scratch-oneshot-fx-pack.rar > /dev/null

In [ ]:
!git clone -b v2 https://github.com/james-trayford/strauss.git

In [ ]:
%cd strauss/examples/

### <u> Demo Ticks </u>

Adding ticks to indicate a fixed interval as a 'plotting' option

In [ ]:
import matplotlib.pyplot as plt
from strauss.sonification import Sonification
from strauss.sources import Objects, Events
from strauss import channels
from strauss.score import Score
import numpy as np
from strauss.generator import Synthesizer
import IPython.display as ipd
import glob
import os
import copy
from pathlib import Path
%matplotlib inline

Let's, mimic the soundfont light-curve notebook, but just use the Synth

In [ ]:
synth = Synthesizer()
synth.load_preset('pitch_mapper')

First let's add ticks to an `Event` sonification

In [ ]:
# pick a soundfont to use

#generator = guitar_sampler
generator = copy.copy(synth)

lightcurve = np.genfromtxt(Path('..', 'data', 'datasets', '55Cancri_lc.dat'))
x = lightcurve[:,0][:]
y = lightcurve[:,1][:]

notes = [["C3","D3","E3","G3","B3","C4","D4","E4","G4","B4","C5","D5","E5","G5","B5"]]
score =  Score(notes, 15)

maps = {'pitch':y,
        'time': x}

system = "mono"

# manually set note properties to get a suitable sound
generator.modify_preset({'note_length':0.03, # hold each note for 0.03 seconds or 30 ms - what if this was 1s?
                         'volume_envelope': {'use':'on',
                                            # A,D,R values in seconds, S sustain fraction from 0-1 that note
                                            # will 'decay' to (after time A+D)
                                            'A':0.01,    # ✏️ Time to fade in note to maximum volume, using 10 ms
                                            'D':0.06,    # ✏️ Time to fall from maximum volume to sustained level (s), irrelevant while S is 1
                                            'S':0.,      # ✏️ fraction of maximum volume to sustain note at while held, 1 implies 100%
                                            'R':0.07}}) # ✏️ Time to fade out once note is released, using 100 ms

# alternatively can avoid setting manually above anf just load the 'staccato' preset
# generator.load_preset('staccato')

# set 0 to 100 percentile limits so the full pitch range is used...
# setting 0 to 101 for pitch means the sonification is 1% longer than
# the time needed to trigger each note - by making this more than 100%
# we give all the notes time to ring out (setting this at 100% means
# the final note is triggered at the momement the sonification ends)
lims = {'time': ('0%','101%'),
        'pitch': ('0%','100%')}

# set up source
sources = Events(maps.keys())
sources.fromdict(maps)
sources.apply_mapping_functions(map_lims=lims)

soni = Sonification(score, sources, generator, system)
soni.render()

# Let's add ticks at day intervals.
# This is input in 'time' or 'time_evo' input units:
# remember if you rescale the time values before
# input you also need to rescale this increment
# e.g. if 'time' is input is in days and increment
# soni.add_ticks(1), there will be a tick for each day
# in the data. If is it's in seconds, soni.add_ticks(120)
# will tick every 2 minutes in the data. optional
# arguments are the duration (seconds) and volume
# (tick_vol, in amplitude fraction).
soni.add_ticks(1., duration=0.04, tick_vol=0.5)

dobj = soni.notebook_display(show_waveform=0)

plt.scatter(x,y, marker='.')
plt.ylabel('Magnitude')
plt.xlabel('Time (Julian Days)')

Now an `Objects` sonification

In [ ]:
# pick a soundfont to use

generator = Synthesizer()
#generator = flute_sampler

generator.modify_preset({'filter':'on'})

# or, just load the 'sustain' preset
# generator.load_preset('sustain')

# we use a 'chord' here to create more harmonic richness (stacking fifths)...
notes = [["E2", "B3"]]
score =  Score(notes, 15)

data = {'pitch':[0,1,2,3],
        'time_evo':[x]*4,
        'cutoff':[y]*4}

lims = {'time_evo': ('0%','100%'),
        'cutoff': ('0%','100%')}

# set up source
sources = Objects(data.keys())
sources.fromdict(data)
plims = {'cutoff': (0.25,0.95)}
sources.apply_mapping_functions(map_lims=lims, param_lims=plims)

soni = Sonification(score, sources, generator, system)
soni.render()

# AGain add ticks at day intervals.
# With the continuous sound we  use a louder, shorter tick.
# lets set it to 0.01s (10 ms). Making it very short
# can also affect it's prominence, and have spectral effects
# (e.g. for 10ms, can't contain frequencies below 1/0.01s = 100 Hz)
soni.add_ticks(1., duration=0.01, tick_vol=1)

dobj = soni.notebook_display(show_waveform=0);
plt.plot(x,y)
plt.ylabel('Magnitude')
plt.xlabel('Time (Julian Days)')

### <u> Multi-format saving via `ffmpeg` </u>

Finally, demo the multiformat saving if wanted, that can be downloaded from the side bar

In [ ]:
soni.save("../../test.wav")
soni.save("../../test.mp3")
soni.save("../../test.aac")

# because we pass to ffmpeg, it can do potentially weird stuff like encode to video
soni.save("../../test.mp4")

In [ ]:
from IPython.display import Audio, Video

# Validate via play-back from file

#Audio("../../test.wav")
#Audio("../../test.mp3")
Audio("../../test.aac")

### <u> Sampler Enhancements </u>

The sampler will now aautomatically read all samples taking any labelled pitch, and if unlabelled, assigning a pitch. This means we can take the 5 sampled notes of the Glockenspiel and play any notes we want.

In [ ]:
from strauss.generator import Sampler

chords = "G#maj13_1"
length = "1m 30s"
score =  Score(chords, length)

datafile = Path("..", "data", "datasets", "stars_paranal.txt")
mapcols =  {'azimuth':1, 'polar':0, 'volume':2, 'time':2, 'pitch':3}

mapvals =  {'azimuth': lambda x : x,
            'polar': lambda x : 90.-x,
            'time': lambda x : x,
            'pitch' : lambda x: -x,
            'volume' : lambda x : (1+np.argsort(x).astype(float))**-0.2}

maplims =  {'azimuth': (0, 360),
            'polar': (0, 180),
            'time': ('0', '104%'),
            'pitch' : ('0', '100%'),
            'volume' : ('0', '100%')}

events = Events(mapcols.keys())
events.fromfile(datafile, mapcols)
events.apply_mapping_functions(mapvals, maplims)### <u> Sampler Enhancements </u>

In [ ]:
soni = Sonification(score, events, sampler, 'stereo')
soni.render()
soni.notebook_display()

We also have an info function to map out the 'keyboard' with samples

NB: These 'aliases' will be also able to be used to call samples directly - makes more sense e.g. with unpitched samples and future `SpeechSynth`.

In [ ]:
sampler = Sampler(Path("..", "data", "samples", "glockenspiels"))
sampler.info()